# Exp1A Formal: D3 + D4 Corrected Marginals — All 8 Corpora

Condition-specific shuffled baselines, 1 shuffle per c (matches original pipeline).
If signal is clear, this is the result. If borderline, increase shuffles on those corpora.

- Probe: Llama base (unsloth/Meta-Llama-3.1-8B)
- Corpora: all 8
- Conditions: intact, D3 (M=50), D4
- ~15 min per corpus per condition, ~6 hrs total

In [ ]:
!pip install -q -U bitsandbytes>=0.46.1 accelerate

import numpy as np
import json, math, os, gc, random, time
from pathlib import Path
from scipy import stats
from scipy.ndimage import uniform_filter1d
from tqdm.auto import tqdm
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

from google.colab import drive
drive.mount('/content/drive')

BASE = Path('/content/drive/MyDrive/LRTIA/Results/Exp1A_formal')
BASE.mkdir(parents=True, exist_ok=True)
DATA = Path('/content/drive/MyDrive/LRTIA/Data')

CORPORA = {
    'wiki_zh': DATA / 'wiki_multilingual/zh_articles.jsonl',
    'wiki_ja': DATA / 'wiki_multilingual/ja_articles.jsonl',
    'wiki_ko': DATA / 'wiki_multilingual/ko_articles.jsonl',
    'wiki_tr': DATA / 'wiki_multilingual/tr_articles.jsonl',
    'wiki_ar': DATA / 'wiki_multilingual/ar_articles.jsonl',
    'wiki_fi': DATA / 'wiki_multilingual/fi_articles.jsonl',
    'buckeye': DATA / 'buckeye_processed/speaker_concatenated.jsonl',
    'french':  DATA / 'french_oral_processed/per_story.jsonl',
}

MODEL_NAME = 'unsloth/Meta-Llama-3.1-8B'
C = 100
TARGET_LEN = 30
TARGET_FRACS = [0.25, 0.50, 0.75]
MIN_BEFORE = C + 10
M = 50
N_SHUFFLES = 1    # start minimal — increase if needed
SEED = 20260429

print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'{N_SHUFFLES} shuffle(s) per c — condition-specific baselines')
print(f'Corpora: {len(CORPORA)}')
print('Setup done')

In [ ]:
# === Load model ===
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.float16),
    device_map='auto'
)
model.eval()
print('Llama loaded')

In [ ]:
# === Functions ===

def split_ctx(ctx, M):
    return ctx[:len(ctx)-M], ctx[len(ctx)-M:]

def d3_swap(ctx, M=50):
    far, near = split_ctx(ctx, M)
    return near + far

def d4_reverse(ctx):
    return list(reversed(ctx))

@torch.no_grad()
def compute_ppl_nll(context_tokens, target_tokens):
    if len(target_tokens) < 2:
        return float('inf'), float('inf')
    full = list(context_tokens) + list(target_tokens)
    ts = len(context_tokens)
    ids = torch.tensor([full], device=model.device)
    out = model(ids)
    logits = out.logits[0]
    nll_sum = 0.0
    cnt = 0
    for i in range(ts, len(full) - 1):
        lp = torch.log_softmax(logits[i], dim=-1)
        nll_sum += -lp[full[i+1]].item()
        cnt += 1
    del out, logits
    torch.cuda.empty_cache()
    if cnt == 0:
        return float('inf'), float('inf')
    mean_nll = nll_sum / cnt
    return math.exp(mean_nll), mean_nll

def compute_curves(cond_ctx, target_tokens):
    """Corrected marginals with condition-specific shuffled baseline."""
    max_c = len(cond_ctx)
    o_ppl, o_nll = [], []
    s_ppl, s_nll = [], []
    
    for c in range(max_c + 1):
        prefix = cond_ctx[-c:] if c > 0 else []
        ppl, nll = compute_ppl_nll(prefix, target_tokens)
        o_ppl.append(ppl); o_nll.append(nll)
        
        if c == 0:
            s_ppl.append(ppl); s_nll.append(nll)
        else:
            rng = random.Random(SEED + c)
            sp_list, sn_list = [], []
            for _ in range(N_SHUFFLES):
                shuf = list(prefix)
                rng.shuffle(shuf)
                sp, sn = compute_ppl_nll(shuf, target_tokens)
                if not math.isinf(sp):
                    sp_list.append(sp); sn_list.append(sn)
            s_ppl.append(np.mean(sp_list) if sp_list else ppl)
            s_nll.append(np.mean(sn_list) if sn_list else nll)
    
    dists = list(range(1, max_c + 1))
    mo = [o_ppl[d-1] - o_ppl[d] for d in dists]
    ms = [s_ppl[d-1] - s_ppl[d] for d in dists]
    delta = [a - b for a, b in zip(mo, ms)]
    mo_n = [o_nll[d-1] - o_nll[d] for d in dists]
    ms_n = [s_nll[d-1] - s_nll[d] for d in dists]
    delta_n = [a - b for a, b in zip(mo_n, ms_n)]
    
    return {
        'distances': dists,
        'ordered_ppl': o_ppl, 'ordered_nll': o_nll,
        'shuffled_ppl': s_ppl, 'shuffled_nll': s_nll,
        'm_ordered': mo, 'm_shuffled': ms, 'delta_ppl': delta,
        'm_ordered_nll': mo_n, 'm_shuffled_nll': ms_n, 'delta_nll': delta_n,
    }

print(f'Functions ready — {N_SHUFFLES} shuffle(s) per c')

In [ ]:
# === Run all 8 corpora ===

CONDS = {
    'intact': lambda ctx: list(ctx),
    'D3_M50': lambda ctx: d3_swap(ctx, M),
    'D4': lambda ctx: d4_reverse(ctx),
}

for corpus_name, corpus_path in CORPORA.items():
    print(f'\n{"="*60}')
    print(f'{corpus_name}')
    print(f'{"="*60}')
    
    if not corpus_path.exists():
        print(f'  NOT FOUND'); continue
    
    docs = []
    with open(corpus_path) as f:
        for line in f:
            docs.append(json.loads(line))
    
    for cond_name, cond_fn in CONDS.items():
        cache = BASE / f'llama_{corpus_name}_{cond_name}.json'
        if cache.exists():
            with open(cache) as f: n = len(json.load(f))
            print(f'  {cond_name}: cached ({n})'); continue
        
        t0 = time.time()
        results = []
        for doc in tqdm(docs, desc=f'{corpus_name}/{cond_name}'):
            full_ids = tokenizer.encode(doc['text'], add_special_tokens=False)
            n = len(full_ids)
            for frac in TARGET_FRACS:
                ts = int(n * frac)
                te = min(ts + TARGET_LEN, n)
                if ts < MIN_BEFORE or te - ts < 5: continue
                ctx = full_ids[ts-C:ts]
                target = full_ids[ts:te]
                r = compute_curves(cond_fn(ctx), target)
                r['doc_id'] = doc.get('doc_id', '')
                r['target_frac'] = frac
                results.append(r)
        
        with open(cache, 'w') as f:
            json.dump(results, f)
        elapsed = time.time() - t0
        print(f'  {cond_name}: {len(results)} results in {elapsed/60:.1f} min')
        
        if results:
            md = np.mean([np.mean(r['delta_ppl']) for r in results])
            print(f'    Mean Δ(ppl): {md:.6f}')

In [ ]:
# === Pass/fail analysis per corpus ===

print(f'\n{"="*70}')
print('FORMAL RESULTS: Corrected Marginals (condition-specific baselines)')
print(f'{"="*70}')

d3_pass = []
d4_pass = []

for corpus_name in CORPORA:
    print(f'\n--- {corpus_name} ---')
    
    data = {}
    for cond in ['intact', 'D3_M50', 'D4']:
        cp = BASE / f'llama_{corpus_name}_{cond}.json'
        if not cp.exists(): continue
        with open(cp) as f: data[cond] = json.load(f)
    
    if 'intact' not in data:
        print('  No intact data'); continue
    
    intact_delta = np.mean([r['delta_ppl'] for r in data['intact']], axis=0)
    
    # --- D3 ---
    if 'D3_M50' in data:
        d3_delta = np.mean([r['delta_ppl'] for r in data['D3_M50']], axis=0)
        
        # Jump at M+1
        jump = d3_delta[M] - d3_delta[M-1] if len(d3_delta) > M else 0
        pre_sd = np.std(d3_delta[:M])
        z = jump / pre_sd if pre_sd > 0 else 0
        jump_pass = jump > 0 and z > 0.5
        
        # Post-jump similarity
        d3_post = d3_delta[M:]
        intact_near = intact_delta[:M]
        intact_far = intact_delta[M:]
        min_len = min(len(d3_post), len(intact_near), len(intact_far))
        if min_len >= 5:
            rho_near, _ = stats.spearmanr(d3_post[:min_len], intact_near[:min_len])
            rho_far, _ = stats.spearmanr(d3_post[:min_len], intact_far[:min_len])
            contrast = rho_near - rho_far
            sim_pass = contrast > 0.3
        else:
            rho_near = rho_far = contrast = 0
            sim_pass = False
        
        d3_overall = jump_pass and sim_pass
        d3_pass.append(d3_overall)
        
        status = 'PASS' if d3_overall else 'FAIL'
        print(f'  D3: jump={jump:.4f} z={z:.2f} ({"ok" if jump_pass else "fail"}), '
              f'contrast={contrast:.3f} ({"ok" if sim_pass else "fail"}) → {status}')
    
    # --- D4 ---
    if 'D4' in data:
        d4_delta = np.mean([r['delta_ppl'] for r in data['D4']], axis=0)
        rev_intact = intact_delta[::-1]
        
        rho_rev, _ = stats.spearmanr(d4_delta, rev_intact)
        rho_fwd, _ = stats.spearmanr(d4_delta, intact_delta)
        contrast = rho_rev - rho_fwd
        
        pass_contrast = contrast > 0.3
        pass_rev = rho_rev >= 0.6
        d4_overall = pass_contrast and pass_rev
        d4_pass.append(d4_overall)
        
        # Floor check
        mean_abs_delta = np.mean(np.abs(d4_delta))
        intact_mag = np.mean(np.abs(intact_delta))
        
        # Magnitude effect
        intact_ppl = np.mean([r['ordered_ppl'] for r in data['intact']], axis=0)
        d4_ppl = np.mean([r['ordered_ppl'] for r in data['D4']], axis=0)
        preserved = (d4_ppl[0] - d4_ppl[-1]) / (intact_ppl[0] - intact_ppl[-1]) if (intact_ppl[0] - intact_ppl[-1]) > 0 else 0
        
        status = 'PASS' if d4_overall else 'FAIL'
        print(f'  D4: rho_rev={rho_rev:.3f} ({"ok" if pass_rev else "fail"}), '
              f'contrast={contrast:.3f} ({"ok" if pass_contrast else "fail"}), '
              f'PPL preserved={preserved:.0%} → {status}')

# === Aggregate ===
print(f'\n{"="*70}')
print('AGGREGATE')
print(f'{"="*70}')
n_d3 = sum(d3_pass)
n_d4 = sum(d4_pass)
print(f'D3: {n_d3}/{len(d3_pass)} pass (threshold: >=6/8)')
print(f'D4: {n_d4}/{len(d4_pass)} pass (threshold: >=6/8)')
print(f'\nCore order-sensitivity: {"PASS" if n_d3 >= 6 and n_d4 >= 6 else "NOT YET"}')
if n_d3 < 6 or n_d4 < 6:
    print('Consider: increase shuffles on borderline corpora, or check floor-limited segments')

In [ ]:
# === Visualization: all 8 corpora ===
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 4, figsize=(28, 10))
axes = axes.flatten()

for idx, corpus_name in enumerate(CORPORA):
    ax = axes[idx]
    for cond, color in [('intact','blue'), ('D3_M50','red'), ('D4','green')]:
        cp = BASE / f'llama_{corpus_name}_{cond}.json'
        if not cp.exists(): continue
        with open(cp) as f: results = json.load(f)
        if not results: continue
        curve = np.mean([r['delta_ppl'] for r in results], axis=0)
        smooth = uniform_filter1d(curve, 5)
        ax.plot(range(1, len(smooth)+1), smooth, color=color, linewidth=2, label=cond)
    ax.axvline(M, color='gray', linestyle=':', alpha=0.5)
    ax.axhline(0, color='gray', linestyle=':', alpha=0.3)
    ax.set_title(corpus_name, fontweight='bold')
    ax.set_xlabel('Distance d')
    if idx % 4 == 0: ax.set_ylabel('Δ_d (corrected marginal)')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.15)

plt.suptitle('Exp1A: D3 + D4 Corrected Marginals — All 8 Corpora (Llama)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(BASE / 'fig_D3_D4_corrected_all.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved')

In [ ]:
# === Also check: do corrected marginals improve D4 Spearman vs raw? ===
# Compare corrected vs raw marginals for D4

RAW_BASE = Path('/content/drive/MyDrive/LRTIA/Results/Exp1_disruption_raw')

print(f'\n{"="*60}')
print('CORRECTED vs RAW: D4 Spearman contrast')
print(f'{"="*60}')
print(f'{"Corpus":<15} {"Raw contrast":>15} {"Corrected contrast":>20}')
print('-' * 53)

for corpus_name in ['wiki_zh', 'buckeye']:
    # Raw (from pilot)
    raw_contrast = '—'
    rp = RAW_BASE / f'llama_{corpus_name}_D4.json'
    rip = RAW_BASE / f'llama_{corpus_name}_intact.json'
    if rp.exists() and rip.exists():
        with open(rp) as f: raw_d4 = json.load(f)
        with open(rip) as f: raw_intact = json.load(f)
        raw_d4_m = np.mean([r['marginals'] for r in raw_d4], axis=0)
        raw_int_m = np.mean([r['marginals'] for r in raw_intact], axis=0)
        rr, _ = stats.spearmanr(raw_d4_m, raw_int_m[::-1])
        rf, _ = stats.spearmanr(raw_d4_m, raw_int_m)
        raw_contrast = f'{rr-rf:.3f}'
    
    # Corrected
    corr_contrast = '—'
    cp = BASE / f'llama_{corpus_name}_D4.json'
    cip = BASE / f'llama_{corpus_name}_intact.json'
    if cp.exists() and cip.exists():
        with open(cp) as f: corr_d4 = json.load(f)
        with open(cip) as f: corr_intact = json.load(f)
        cd4 = np.mean([r['delta_ppl'] for r in corr_d4], axis=0)
        ci = np.mean([r['delta_ppl'] for r in corr_intact], axis=0)
        cr, _ = stats.spearmanr(cd4, ci[::-1])
        cf, _ = stats.spearmanr(cd4, ci)
        corr_contrast = f'{cr-cf:.3f}'
    
    print(f'{corpus_name:<15} {raw_contrast:>15} {corr_contrast:>20}')